In [ ]:
import json
import google.generativeai as genai
from time import sleep
import re
import os

# === Cấu hình API Key Gemini ===
genai.configure(api_key="")  # THAY API_KEY CỦA BẠN

output_file = "/kaggle/working/unknown_data.jsonl"
log_file = "/kaggle/working/unknown_log.txt"
batch_size = 6
target_samples = 3000
temperature = 0.7
max_retry = 3

# === DANH SÁCH MODEL FALLBACK ===
model_priority_list = [
    'gemini-2.5-flash-lite',
    'gemini-2.5-flash', 
    'gemini-2.5-pro',
    'gemini-2.0-flash',
    'gemini-2.0-flash-lite',
]

max_output_tokens = 8192

def get_available_model():
    """Lấy model đầu tiên khả dụng trong danh sách"""
    for model_name in model_priority_list:
        try:
            model = genai.GenerativeModel(model_name)
            print(f"✅ Sử dụng model: {model_name}")
            return model_name
        except Exception as e:
            print(f"❌ Model {model_name} không khả dụng: {e}")
            continue
    print(f"⚠️  Cảnh báo: Không có model nào khả dụng, thử dùng {model_priority_list[0]}")
    return model_priority_list[0]

def clean_json_response(text):
    """Làm sạch response JSON từ API"""
    cleaned = re.sub(r'```json\s*', '', text)
    cleaned = re.sub(r'\s*```', '', cleaned)
    cleaned = cleaned.strip()
    
    cleaned = re.sub(r'^[^{[]*', '', cleaned)
    cleaned = re.sub(r'[^}\]]*$', '', cleaned)
    
    return cleaned

def repair_truncated_json(json_str):
    """Sửa JSON bị cắt ngang"""
    if not json_str.strip():
        return json_str
        
    open_braces = json_str.count('{')
    close_braces = json_str.count('}')
    open_brackets = json_str.count('[')
    close_brackets = json_str.count(']')
    
    repaired = json_str
    
    if open_braces > close_braces:
        repaired += '}' * (open_braces - close_braces)
    
    if open_brackets > close_brackets:
        repaired += ']' * (open_brackets - close_brackets)
    
    return repaired

def extract_and_parse_json(response_text):
    """Trích xuất và parse JSON từ response text"""
    print(f"🔧 Đang xử lý response dài {len(response_text)} chars...")
    
    cleaned_text = clean_json_response(response_text)
    print(f"🔧 Sau khi làm sạch: {len(cleaned_text)} chars")
    
    # Thử parse trực tiếp
    try:
        json_data = json.loads(cleaned_text)
        print("✅ Parse trực tiếp thành công")
        return json_data
    except json.JSONDecodeError as e:
        print(f"❌ Parse trực tiếp thất bại: {e}")
    
    # Tìm JSON array bằng regex
    array_pattern = r'\[\s*\{[\s\S]*?\}\s*\]'
    array_matches = re.findall(array_pattern, cleaned_text, re.DOTALL)
    
    if array_matches:
        print(f"✅ Tìm thấy {len(array_matches)} JSON arrays bằng regex")
        json_str = max(array_matches, key=len)
        json_str_repaired = repair_truncated_json(json_str)
        print(f"🔧 Đã sửa JSON (từ {len(json_str)} lên {len(json_str_repaired)} chars)")
        
        try:
            json_data = json.loads(json_str_repaired)
            print("✅ Parse JSON đã sửa thành công")
            return json_data
        except json.JSONDecodeError as e2:
            print(f"❌ Vẫn lỗi JSON sau khi sửa: {e2}")
    
    # Tìm thủ công
    start_idx = cleaned_text.find('[')
    end_idx = cleaned_text.rfind(']')
    
    if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
        json_str = cleaned_text[start_idx:end_idx+1]
        print(f"✅ Tìm thấy JSON bằng manual extraction ({len(json_str)} chars)")
        json_str_repaired = repair_truncated_json(json_str)
        
        try:
            json_data = json.loads(json_str_repaired)
            print("✅ Parse JSON manual thành công")
            return json_data
        except json.JSONDecodeError as e3:
            print(f"❌ Lỗi parse JSON manual: {e3}")
    
    raise ValueError("Không thể trích xuất JSON từ response")

def generate_unknown_batch(batch_size):
    """Tạo batch data 'không biết' trong phạm vi lịch sử"""
    prompt = f"""
HÃY TẠO {batch_size} MẪU HỘI THOẠI JSON CHO CHATBOT LỊCH SỬ VIỆT NAM:

YÊU CẦU:
- QUESTION: Câu hỏi về lịch sử Việt Nam nhưng về sự kiện/nhân vật KHÔNG TỒN TẠI hoặc quá chi tiết không có trong tài liệu
- ANSWER: Thừa nhận không biết + gợi ý có thể sự kiện không tồn tại
- Đa dạng loại câu hỏi không tồn tại

ĐỊNH DẠNG JSON:
[
  {{
    "messages": [
      {{"role": "system", "content": "Bạn là chuyên gia lịch sử Việt Nam."}},
      {{"role": "user", "content": "QUESTION"}},
      {{"role": "assistant", "content": "ANSWER"}}
    ]
  }}
]

QUAN TRỌNG: CHỈ TRẢ VỀ JSON, KHÔNG THÊM VĂN BẢN NÀO KHÁC.
"""

    current_model = get_available_model()
    
    for attempt in range(1, max_retry + 1):
        try:
            print(f"🔄 Attempt {attempt} với model {current_model}...")
            
            model = genai.GenerativeModel(current_model)
            response = model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=temperature,
                    max_output_tokens=max_output_tokens,
                    top_p=0.8
                )
            )
            
            response_text = response.text.strip()
            print(f"📄 Raw response length: {len(response_text)} chars")
            
            if len(response_text) > 300:
                print(f"📄 First 300 chars: {response_text[:300]}...")
            
            # Xử lý JSON
            json_data = extract_and_parse_json(response_text)
            
            # Kiểm tra cấu trúc
            if not isinstance(json_data, list):
                raise ValueError("Kết quả không phải là list")
                
            print(f"✅ Đã tạo được {len(json_data)} samples")
                
            for i, item in enumerate(json_data):
                if "messages" not in item:
                    raise ValueError(f"Thiếu key 'messages' trong item {i}")
                if not isinstance(item["messages"], list):
                    raise ValueError(f"'messages' trong item {i} không phải là list")
                if len(item["messages"]) < 2:
                    raise ValueError(f"Hội thoại {i} quá ngắn")
                    
            return json_data
            
        except Exception as e:
            print(f"❌ Lỗi attempt {attempt} với model {current_model}: {str(e)[:200]}")
            
            if attempt < max_retry:
                next_model_index = (model_priority_list.index(current_model) + 1) % len(model_priority_list)
                current_model = model_priority_list[next_model_index]
                print(f"🔄 Chuyển sang model: {current_model}")
                sleep(3)
            else:
                print(f"💥 Đã thử tất cả {max_retry} lần")
                return []

def create_fallback_unknown_batch(batch_size):
    """Tạo fallback data khi API thất bại"""
    fallback_data = []
    fake_events = [
        "trận chiến Đồng Quan năm 1234",
        "vua Lý Thần Tông thứ 5", 
        "khởi nghĩa Ba Đình lần thứ 2",
        "hòa ước Nhâm Tuất 1863",
        "chiến thắng sông Hồng năm 1785",
        "triều đại nhà Lê thứ 3",
        "cuộc kháng chiến chống Mông-Nguyên lần thứ 4",
        "vua Trần Anh Tông đời thứ 6",
        "trận đánh tại núi Bạch Mã năm 1420",
        "hiệp ước Phú Xuân 1750"
    ]
    
    unknown_responses = [
        "Tôi không có thông tin về sự kiện này. Có thể sự kiện/nhân vật này không tồn tại trong lịch sử Việt Nam.",
        "Tôi chưa được cung cấp thông tin về sự kiện này. Có thể bạn đã nhầm lẫn với một sự kiện khác.",
        "Thông tin này không có trong các tài liệu lịch sử chính thống của Việt Nam.",
        "Tôi không tìm thấy dữ liệu về sự kiện này. Bạn có thể kiểm tra lại thông tin không?",
        "Theo kiến thức của tôi, sự kiện này không tồn tại trong lịch sử Việt Nam."
    ]
    
    for i in range(batch_size):
        event = fake_events[i % len(fake_events)]
        response = unknown_responses[i % len(unknown_responses)]
        
        messages = [
            {"role": "system", "content": "Bạn là chuyên gia lịch sử Việt Nam."},
            {"role": "user", "content": f"{event} diễn ra như thế nào?"},
            {"role": "assistant", "content": response}
        ]
        fallback_data.append({"messages": messages})
    
    return fallback_data

# === XỬ LÝ CHÍNH ===
processed_count = 0

print(f"🎯 Bắt đầu tạo {target_samples} samples data 'không biết'...")

with open(output_file, "w", encoding="utf-8") as f_out, \
     open(log_file, "w", encoding="utf-8") as f_log:

    while processed_count < target_samples:
        current_batch_size = min(batch_size, target_samples - processed_count)
        print(f"\n🚀 Đang tạo batch {processed_count + 1} đến {processed_count + current_batch_size}...")
        
        batch_data = generate_unknown_batch(current_batch_size)
        
        if not batch_data:
            print("🔄 Sử dụng fallback...")
            batch_data = create_fallback_unknown_batch(current_batch_size)
        
        for data in batch_data:
            f_out.write(json.dumps(data, ensure_ascii=False) + "\n")
            processed_count += 1
        
        f_log.write(f"Đã tạo {processed_count}/{target_samples} samples\n")
        print(f"✅ Đã tạo {processed_count}/{target_samples} samples")
        
        if processed_count < target_samples:
            sleep(5)  # Chờ giữa các batch để tránh rate limit

print(f"\n🎉 Hoàn tất! Đã tạo {processed_count} samples data 'không biết'.")
print(f"📁 Output: {output_file}")
print(f"📋 Log: {log_file}")

🎯 Bắt đầu tạo 3000 samples data 'không biết'...

🚀 Đang tạo batch 1 đến 6...
✅ Sử dụng model: gemini-2.5-flash-lite
🔄 Attempt 1 với model gemini-2.5-flash-lite...
📄 Raw response length: 4037 chars
📄 First 300 chars: ```json
[
  {
    "messages": [
      {"role": "system", "content": "Bạn là chuyên gia lịch sử Việt Nam."},
      {"role": "user", "content": "Chi tiết về trận chiến Đống Đa năm 1789, cụ thể là số lượng voi chiến mà quân Tây Sơn đã sử dụng và cách chúng được huấn luyện để tấn công bằng lửa?"},
     ...
🔧 Đang xử lý response dài 4037 chars...
🔧 Sau khi làm sạch: 4025 chars
✅ Parse trực tiếp thành công
✅ Đã tạo được 6 samples
✅ Đã tạo 6/3000 samples

🚀 Đang tạo batch 7 đến 12...
✅ Sử dụng model: gemini-2.5-flash-lite
🔄 Attempt 1 với model gemini-2.5-flash-lite...
📄 Raw response length: 5147 chars
📄 First 300 chars: ```json
[
  {
    "messages": [
      {"role": "system", "content": "Bạn là chuyên gia lịch sử Việt Nam."},
      {"role": "user", "content": "Hãy kể cho tôi nghe 